<a href="https://colab.research.google.com/github/Chosencodes/Cardiac_Heart_Detection/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install albumentations -q

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
labels_path   = "/content/drive/MyDrive/05-Detection/rsna_heart_detection.csv"
train_patients = "/content/drive/MyDrive/05-Detection/train_subjects.npy"
val_patients   = "/content/drive/MyDrive/05-Detection/val_subjects.npy"
train_root     = "/content/drive/MyDrive/cardiac-heart-detection/Processed-Heart-Detection/train/"
val_root       = "/content/drive/MyDrive/cardiac-heart-detection/Processed-Heart-Detection/val/"

In [ ]:
dataset_code = '''
import torch
import numpy as np
import pandas as pd
import albumentations as A
from pathlib import Path
import os

class CardiacDataset(torch.utils.data.Dataset):
  def __init__(self, path_to_labels_csv, patients, root_path, augs=None):
    self.labels = pd.read_csv(path_to_labels_csv)
    self.patients = np.load(patients)
    self.root_path = Path(root_path)
    self.augment = augs

  def __len__(self):
    return len(self.patients)

  def __getitem__(self, idx):
    patient = self.patients[idx]
    data = self.labels[self.labels["name"] == patient]

    x_min = data["x0"].item()
    y_min = data["y0"].item()
    x_max = x_min + data["w"].item()
    y_max = y_min + data["h"].item()

    file_path = self.root_path / patient / f"{patient}.npy"
    img = np.load(str(file_path)).astype(np.float32)

    if self.augment:
        img_uint8 = (img * 255).clip(0, 255).astype(np.uint8)
        img_uint8 = np.expand_dims(img_uint8, axis=-1)

        transformed = self.augment(
            image=img_uint8,
            bboxes=[[x_min, y_min, x_max, y_max]],
            labels=["heart"]
        )

        img = transformed["image"].squeeze(-1).astype(np.float32) / 255.0
        if transformed["bboxes"]:
            x_min, y_min, x_max, y_max = transformed["bboxes"][0]

    img = (img - 0.494) / 0.253
    img = torch.tensor(img).unsqueeze(0)
    bbox = torch.tensor([x_min, y_min, x_max, y_max])

    return img, bbox
'''

with open("/content/dataset.py", "w") as f:
    f.write(dataset_code)

In [ ]:
import sys
if "dataset" in sys.modules: sys.modules.pop("dataset")
from dataset import CardiacDataset

# **Validate**

In [ ]:
import sys
if "dataset" in sys.modules: sys.modules.pop("dataset")
from dataset import CardiacDataset

In [ ]:
train_augs = A.Compose([
    A.RandomGamma(p=0.5),
    A.Affine(scale = (0.8, 1.2),rotate = (-10, 10),translate_px = {"x": (-10, 10), "y": (-10, 10)}),
], bbox_params=A.BboxParams(format = "pascal_voc",label_fields = ["labels"], clip = True
))


In [ ]:
dataset = CardiacDataset(labels_path,train_patients,train_root,)

In [ ]:
import shutil
shutil.copy(
    "/content/drive/MyDrive/cardiac-heart-detection/preprocess.ipynb",
    "/content/preprocess.ipynb"
)
print("done")

In [ ]:
img, bbox = dataset[6]

fig, axis = plt.subplots(1,1)
axis.imshow(img[0],cmap="bone")
rect = patches.Rectangle((bbox[0],bbox[1]), bbox[2]-bbox[0],bbox[3]-bbox[1],edgecolor="r",facecolor="none")
axis.add_patch(rect)

In [ ]:
img, bbox = dataset[300]

fig, axis = plt.subplots(1,1)
axis.imshow(img[0],cmap="bone")
rect = patches.Rectangle((bbox[0],bbox[1]), bbox[2]-bbox[0],bbox[3]-bbox[1],edgecolor="r",facecolor="none")
axis.add_patch(rect)